In [76]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 1. 패키지 설치

In [3]:
%pip install -q cohere

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip show cohere

Name: cohere
Version: 7.2.0
Summary: 
Home-page: 
Author: 
Author-email: 
License: MIT
Location: c:\users\mbc\anaconda3\envs\llm\lib\site-packages
Requires: fastavro, httpx, pydantic, pydantic-core, requests, tokenizers, types-requests, typing_extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


# 1. 패키지 설치 및 환경
- cohere 회원가입 후 api_key를 추가 (COHERE_API_KEY=xxx)
- https://docs.cohere.com/docs/rerank 에서 rerank-multilingual-v3.0 모델을 이용한 rerank

# 2. 환경변수 설정

In [7]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)  # 커널을 재시작하지 않아도 환경변수 다시 가져오기
#print(OPEN_API_KEY[:2],PINECONE_API_KEY[:2],UPSTAGE_API_KEY[:2])

OPENAI_LLM_MODEL = "gpt-4o-mini" #사용
# UPSTAGE_LLM_MODEL = "solar-pro4"
# OPENAI_EMBEDDING_MODEL = "text-embedding-3-large" # 차원수 3072
UPSTAGE_EMBEDDING_MODEL = "embedding-query" # 차원수 4096 
PINECONE_INDEX_NAME = "better-rag-index"

COHERE_API_KEY = os.getenv('COHERE_API_KEY')
COHERE_API_KEY[:2]

'ta'

# 3. category 추출하기
- chunk뿐 만 아니라, query의 category추출

In [8]:
def get_category():
    return {
        '납세의무': {
            '납세의무': 3, '거주자': 3, '비거주자': 3, '납세의무자': 3, 
            '원천징수': 2, '원천징수의무자': 2, '공동사업자': 2, 
            '상속인': 2, '증여자': 2, '신탁재산': 2
        },

        '세율계산': {
            '세율': 3, '소득세': 3, '과세표준': 3, '산출세액': 3, '세액': 3,
            '결정세액': 2, '세액계산': 2, '기본세율': 2, '세율적용': 2,
            '누진세율': 2, '종합소득세': 2
        },

        '근로소득': {
            '근로소득': 3, '총급여': 3, '급여': 3, '연봉': 3, '임금': 3,
            '근로소득금액': 2, '총급여액': 2, '상여': 2, '수당': 2,
            '봉급': 2, '직장인': 2
        },

        '사업소득': {
            '사업소득': 3, '총수입금액': 3, '필요경비': 3,
            '사업자': 2, '사업소득금액': 2, '결손금': 2, '이월결손금': 2,
            '주택임대소득': 2, '공동사업': 2
        },

        '이자배당': {
            '이자소득': 3, '배당소득': 3, '예금이자': 2, '채권': 2,
            '의제배당': 2, '배당세액공제': 2, '분리과세이자소득': 2,
            '분리과세배당소득': 2
        },

        '양도소득': {
            '양도소득': 3, '자산양도': 2, '부동산양도': 2, '주식양도': 2,
            '양도차익': 2, '취득가액': 2, '양도가액': 2, '양도소득금액': 2
        },

        '연금소득': {
            '연금소득': 3, '연금계좌': 3, '연금저축': 3,
            '퇴직연금': 2, '공적연금': 2, '사적연금': 2, '연금보험': 2,
            '연금수령': 2
        },

        '기타소득': {
            '기타소득': 3, '가상자산': 3, '가상자산소득': 2,
            '상금': 2, '보상금': 2, '종교인소득': 2, '원고료': 2,
            '복권': 1, '당첨금': 1, '발명보상금': 1
        },

        '공제감면': {
            '공제': 3, '소득공제': 3, '세액공제': 3,
            '기본공제': 2, '인적공제': 2, '특별공제': 2, '추가공제': 2,
            '근로소득공제': 2, '연금소득공제': 2, '퇴직소득공제': 2,
            '연금계좌세액공제': 2, '감면': 2
        },

        '비과세': {
            '비과세': 3, '비과세소득': 3, '면제': 2, '세액감면': 2,
            '소득세면제': 2, '복무급여': 1, '실업급여': 1, 
            '출산휴가급여': 1, '장학금': 1
        },

        '신고납부': {
            '신고': 3, '확정신고': 3, '과세표준확정신고': 3, 
            '납부': 3, '중간예납': 2, '가산세': 2,
            '신고기한': 2, '납부기한': 2, '납세지': 2
        },

        '과세기간': {
            '과세기간': 3, '과세연도': 3, '사업연도': 2, 
            '과세기간종료일': 2
        },

        '과세방식': {
            '종합과세': 3, '분리과세': 3, '합산과세': 2, 
            '종합소득과세표준': 2, '분리과세소득': 2, 
            '금융투자소득': 2
        },

        '장부기장': {
            '장부': 3, '복식부기': 3, '간편장부': 2, '기장': 2,
            '장부기록': 2, '증명서류': 2, '기장세액공제': 2
        }
    }



In [9]:
def categorize_content(content, top_k=None):
    """
    content의 카테고리를 분류 - 점수 기반으로 모든 카테고리를 점수순으로 반환(top_k만큼)
        Parameters: 
        - content : 카테고리를 분류할 텍스트
        - top_k : 상위 몇개까지의 카테고리를 반환할 지 정수나 None(None이면 모든 카테고리를 반환)
        Returns : 
        - 카테고리 리스트(점수순) ex {'장부기장':30, '과세방식':45}=>['과세방식', '장부기장']
    
    """
    category_keywords = get_category()
    category_scores = {}
    # 각 카테고리별 점수 계산
    for category, weighted_keywords in category_keywords.items():
        #print(category, weighted_keywords)
        score = 0
        for keyword, weight in weighted_keywords.items():
            #print(keyword,weight)
            count =content.count(keyword) # content에 keyword가 나오는 횟수
            score += count * weight
        if score>0:
            category_scores[category] =score
    #print(category_scores)
    sorted_categories = sorted(category_scores.items(), key=lambda item:item[1], reverse=True)# 내림차순
    #print(sorted_categories)
    all_categories =[category[0] for category in sorted_categories]
    #print(all_categories)
#매칭되는 카테고리가 없을 경우 ["기타"]를 반환
    if not all_categories:
        all_categories = ['기타']
    #print(all_categories)
    # top k 가 None이 아닌경우 
    if top_k is not None:
        return all_categories[:top_k]
    return all_categories    
    

In [12]:
content="""원천징수하는 소득세의 납세지는 다음 각 호에 따른다. <개정 2012. 1. 1., 2023. 12. 31.>
1. 원천징수하는 자가 거주자인 경우: 그 거주자의 주된 사업장 소재지. 다만, 주된 사업장 외의 사업장에서 원천징수를 하는 경우에는 그 사업장의 소재지, 사업장이 없는 경우에는 그 거주자의 주소지 또는 거소지로 한다.
2. 원천징수하는 자가 비거주자인 경우: 그 비거주자의 주된 국내사업장 소재지. 다만, 주된 국내사업장 외의 국내사업장에서 원천징수를 하는 경우에는 그 국내사업장의 소재지, 국내사업장이 없는 경우에는 그 비거주자의 거류지(居留地) 또는 체류지로 한다.
3. 원천징수하는 자가 법인인 경우: 그 법인의 본점 또는 주사무소의 소재지
4. 원천징수하는 자가 법인인 경우로서 그 법인의 지점, 영업소, 그 밖의 사업장이 독립채산제(獨立採算制)에 따라 독자적으로 회계사무를 처리하는 경우: 제3호에도 불구하고 그 사업장의 소재지(그 사업장의 소재지가 국외에 있는 경우는 제외한다). 다만, 대통령령으로 정하는 경우에는 그 법인의 본점 또는 주사무소의 소재지를 소득세 원천징수세액의 납세지로 할 수 있다.
장부 장부 장부"""
#content='아무것도 없다'
print(categorize_content(content))
print(categorize_content("연봉 5천만원인 직장인의 소득세는 얼마예요?"))

['납세의무', '세율계산', '장부기장', '신고납부']
['근로소득', '세율계산']


# 4. langchain 답변 생성

In [14]:
from langchain_openai import ChatOpenAI
from langchain_upstage import UpstageEmbeddings
# from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [15]:
# 1. 임베딩 & LLM 초기화
llm = ChatOpenAI(model=OPENAI_LLM_MODEL) # 수정가능
embedding = UpstageEmbeddings(model=UPSTAGE_EMBEDDING_MODEL) # pinecone에 저장시 사용했던 임베딩 모델 유지해야함

# 2. vector DB load
# pc = Pinecone()
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=PINECONE_INDEX_NAME
)

In [73]:
# 3. 키워드 사전 활용하여 질문 표준화
keyword_dict  = [
    "사람을 나타내는 표현 -> 거주자",
    "직장인 -> 근로소득이 있는 거주자",  
    "월급쟁이 -> 근로소득이 있는 거주자",
    "회사원 -> 근로소득이 있는 거주자",
    "연봉 -> 종합소득",
    "월급 -> 근로소득",
    "세금 -> 소득세",
    "천 ->000"
    "공제받다 -> 공제를 적용받다",
    "얼마나 내야하나 -> 세액은 얼마인가",
    "계산해줘 -> 계산하면 얼마인가"
]
prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만 리턴해 주세요.
사전 : {keyword_dict}
질문 : {{question}}""")
# llm.invoke(prompt.invoke({'question':"월급이 높은 남자가 있습니다"}))
keyword_chain = prompt | llm | StrOutputParser()      # 이 llm 은 괜찮은 모델사용이 필요하다 2.4b이상 
query = "연봉 5000만원인 회사원의 소득세는 얼마예요?"
keyword_chain.invoke({'question':query})
#keyword_chain.invoke(query)
normalized_query = keyword_chain.invoke({'question':query})
normalized_query

'종합소득 5000만원인 근로소득이 있는 거주자의 소득세는 세액은 얼마인가요?'

In [18]:
#4. Retriever 생성
retriever = vectorStore.as_retriever(
    search_type = 'similarity',
    search_kwargs ={
        'k':10,
        'filter': {'category':{'$in':categorize_content(normalized_query)}}  # ['세율계산', '근로소득', '납세의무']  中 하나라도 포함된 우선순위 10개
    }
)
documents=retriever.invoke(normalized_query)
documents

[Document(id='1b1eaa21-3838-4de8-b2e5-756dd01487ed', metadata={'article': '「43조, 51조, 54조, 55조, 59조」', 'category': ['세율계산', '공제감면', '사업소득', '납세의무', '과세방식', '과세기간'], 'chunk_id': 'chunk_043', 'source': 'tax_docx/소득세법_with_markdown.docx', 'title': '소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 주된 사업자의 합산 과세 대상에 포함되며, 해당 금액 한도 내에서 공제 가능. 세율 구조는 과세표준 구간별 차등 적용되며, 거주자의 종합소득 및 퇴직소득에 대한 세액 계산 방식 명시.'}, page_content='[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을

In [19]:
print('순위재조정 전의 관련 문서')
for i, doc in enumerate(documents):
    chunk_id = doc.metadata.get('chunk_id')
    category = doc.metadata.get('category')
    title = doc.metadata.get('title')
    article = doc.metadata.get('article','조항없음')
    print(f'{i+1}.{chunk_id}:{title[:50]}{category[:1]}{article[:5]}')

순위재조정 전의 관련 문서
1.chunk_043:소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 ['세율계산']「43조,
2.chunk_037:소득세 공제 규정 요약 - 일용근로자 공제: 1일 최대 15만원 공제 2018년 개정. -['공제감면']「22조,
3.chunk_048:소득세법 주요 조항 요약: - 근로소득세액공제: 총급여액에 따라 차등 공제 최대 한도 명시['근로소득']「59조,
4.chunk_036:소득공제 및 세액공제 특례 - 결손금 및 이월결손금 공제: 과세기간 결손금 발생 시 먼저 ['공제감면']「4조, 
5.chunk_049:연금계좌세액공제 - 종합소득자 대상, 연금계좌 납입액의 일정 비율 공제 근로소득자 기준: ['연금소득']「59조,
6.chunk_009:종합소득세 과세표준 계산 - 종합소득금액 계산 시 이자, 배당 등 다양한 소득 합산 후 공['이자배당']「2조, 
7.chunk_053:소득세법 공제 및 감면 규정 요약 - 공제 신청 요건: 거주자 신청 시 제1항제4항 적용.['공제감면']「6조, 
8.chunk_050:소득세 공제 항목: 장애인 의료비, 특정 연령대 및 질환자 의료비, 난임시술 비용, 교육비['근로소득']「2조」
9.chunk_010:분리과세 및 분리연금소득 규정 - 분리과세 기타소득: 특정 조건 하의 연금외수령소득, 복권['세율계산']「2조, 
10.chunk_055:종합소득세 세액 계산 특례 거주자의 종합소득세 산출 시, 이자소득 및 배당소득의 합산과세 ['세율계산']「16조,


## title+page_content

In [20]:
documents[0]

Document(id='1b1eaa21-3838-4de8-b2e5-756dd01487ed', metadata={'article': '「43조, 51조, 54조, 55조, 59조」', 'category': ['세율계산', '공제감면', '사업소득', '납세의무', '과세방식', '과세기간'], 'chunk_id': 'chunk_043', 'source': 'tax_docx/소득세법_with_markdown.docx', 'title': '소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 주된 사업자의 합산 과세 대상에 포함되며, 해당 금액 한도 내에서 공제 가능. 세율 구조는 과세표준 구간별 차등 적용되며, 거주자의 종합소득 및 퇴직소득에 대한 세액 계산 방식 명시.'}, page_content='[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 

# title+page_content 기반 rerank함수(cohere)

In [46]:
import cohere
def rerank_by_title(query:str, documents:list, top_k:int=3) -> list:
    '''
    cohere의 rerank API를 사용한 documents 재정렬
    Args:
    - query : 사용자 질문
    - documents : retriever로 검색된 Document 객체 list
    - top_k : 재정렬 후 반환될 문서 갯수 Document
    Returns:
    - title 및 page_content 유사도 기준으로 재정렬된 Document list(0~top_k앞까지) 
    '''
    # Document 객체에서 title과 page_content추출
    docs_text = [doc.metadata.get('title')+" "+doc.page_content for doc in documents]
    # cohere 클라이언트 객체 생성
    co = cohere.Client(
                        #api_key=COHERE_API_KEY
                        )
    #cohere rerank API 호출
    results = co.rerank(
                    model ='rerank-multilingual-v3.0', 
                    query = query,
                    documents = docs_text,
                    top_n = top_k
                        )
    # idx = [result.index for result in results.results] 상위 3개 index 확인함
    #print(idx)
    rerank_docs = [ documents[r. index] for r in results.results]
    return rerank_docs

In [51]:
docs = rerank_by_title(normalized_query,documents,3)

In [52]:
print('순위재조정 후의 관련 문서')
for i, doc in enumerate(docs):
    chunk_id = doc.metadata.get('chunk_id')
    category = doc.metadata.get('category')
    title = doc.metadata.get('title')
    article = doc.metadata.get('article','조항없음')
    print(f'{i+1}.{chunk_id}:{title[:50]}{category[:1]}{article[:5]}')

순위재조정 후의 관련 문서
1.chunk_043:소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 ['세율계산']「43조,
2.chunk_048:소득세법 주요 조항 요약: - 근로소득세액공제: 총급여액에 따라 차등 공제 최대 한도 명시['근로소득']「59조,
3.chunk_009:종합소득세 과세표준 계산 - 종합소득금액 계산 시 이자, 배당 등 다양한 소득 합산 후 공['이자배당']「2조, 


In [53]:
#5. 프롬프트 템플릿
template =f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 반드시 모른다고 답하세요
질문 : {{query}}
문맥 : {{context}}
답변 : """
prompt =ChatPromptTemplate.from_template(template)
# 6. 검색된 documents를 텍스트로 변환하는 함수 list -> page_content만 받자
def format_documents(retrieved_docs):
    return '\n--\n'.join([doc.page_content.replace('\n\n',' ') for doc in retrieved_docs])
# 7. RAG chain 구성(LCEL방식) 
rag_chain = (
    {
        'context': retriever | format_documents,
        'query' : RunnablePassthrough() #질문만 그대로 전달 가능
    } 
    |prompt
    |llm
    |StrOutputParser()
)
final_chain = keyword_chain | rag_chain

In [62]:
# 8. 실행
query = "연봉 5천만원인 직장인의 소득세는 얼마예요?"
result= final_chain.invoke(normalized_query)

In [63]:
result

'종합소득 5000만원인 근로소득이 있는 거주자의 소득세 세액을 계산해보겠습니다.\n\n종합소득 5000만원의 경우, 세율은 다음과 같습니다:\n- 1,400만원 이하: 6%\n- 1,400만원 초과 5,000만원 이하: 84만원 + (1,400만원을 초과하는 금액의 15%)\n\n5000만원의 세액을 계산하면:\n1. 1,400만원에 대한 세액: \\(1,400만원 \\times 6\\% = 84만원\\)\n2. 3,600만원(5,000만원 - 1,400만원)에 대한 세액: \\(3,600만원 \\times 15\\% = 540만원\\)\n\n따라서 종합소득산출세액은:\n- \\(84만원 + 540만원 = 624만원\\)\n\n결론적으로, 종합소득 5000만원인 근로소득이 있는 거주자의 소득세 세액은 **624만원**입니다.'

# 참조조항 포함

In [65]:
docs = retriever.invoke(keyword_chain.invoke(normalized_query))
docs[0]

Document(id='1b1eaa21-3838-4de8-b2e5-756dd01487ed', metadata={'article': '「43조, 51조, 54조, 55조, 59조」', 'category': ['세율계산', '공제감면', '사업소득', '납세의무', '과세방식', '과세기간'], 'chunk_id': 'chunk_043', 'source': 'tax_docx/소득세법_with_markdown.docx', 'title': '소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이 주된 사업자의 합산 과세 대상에 포함되며, 해당 금액 한도 내에서 공제 가능. 세율 구조는 과세표준 구간별 차등 적용되며, 거주자의 종합소득 및 퇴직소득에 대한 세액 계산 방식 명시.'}, page_content='[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 

In [67]:
def extract_articles_from_docs(documents):
    """
    검색된 문서 documents에서 article 정보를 추출하여 리스트로 반환    
    Args:
    - documents : retriever에서 반환된 Document 객체 리스트 
    Return : 
      중복 제거된 조항 리스트를 문자로 바꿔서 반환 (ex. ['소득세법 제4조, 제60조'])
    """
    articles = []
    for doc in documents:
        # metadata에서 article 정보 추출
        article =doc.metadata.get('article', '조항없음')
     #   if article!='조항없음':
        article=article.replace('「','').replace('」','')
        article_list=article.split(', ')
        articles.extend(article_list)
     # 중복제거 및 정렬
    unique_article=list(set(articles))
    unique_article.sort(key=lambda x : int(x[:-1]))
    final_article = ["제"+a for a in unique_article if article !='조항없음']
    result = '소득세법'+','.join(final_article)
    return result
    print(final_article)

In [68]:
extract_articles_from_docs(docs)

'소득세법제2조,제4조,제6조,제12조,제13조,제14조,제15조,제16조,제17조,제19조,제20조,제21조,제22조,제24조,제26조,제27조,제29조,제31조,제35조,제37조,제39조,제41조,제43조,제45조,제46조,제47조,제48조,제49조,제50조,제51조,제52조,제54조,제55조,제58조,제59조,제62조,제64조,제65조,제77조,제91조,제95조,제122조,제127조,제129조,제133조,제134조,제146조,제156조,제160조'

In [70]:
# 재사용이 가능한 함수 만들기
def ask_with_reference(query:str, k:int=10, top_k:int=3):
    """
    수정된 query, 답변, 참고조항을 함께 반환하는 함수 
    Args:
    - query : 질문
    - k : 검색할 문서의 수(기본값: 10)
    - top_k : 재배열된 이후 사용될 문서의 수(기본값 : 3)
    """
    # query 표준화  (키워드 딕셔너리를 통해서 바꿈)
    normalized_query = keyword_chain.invoke({'question':query})
    # retirever 생성
    retriever = vectorStore.as_retriever(
                search_type='similarity',
                search_kwargs = {'k':k,
                                'filter':{'category':{'$in':categorize_content(normalized_query)}}}
    )
    # 검색된 문서
    retrievered_docs = retriever.invoke(normalized_query)
    #Rerank된 문서 추출
    reranked_docs= rerank_by_title(query=normalized_query,
                                  documents=retrievered_docs,
                                  top_k=top_k)
    
    # 참고 조항 추출
    referenced_articles =extract_articles_from_docs(reranked_docs)
    # LLM 답변 생성
    template =f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 반드시 모른다고 답하세요
    질문 : {{query}}
    문맥 : {{context}}
    답변 : """
    prompt =ChatPromptTemplate.from_template(template)
    prompt_chain = prompt | llm | StrOutputParser()    
    answer = prompt_chain.invoke({
                        'query': normalized_query,
                        'context': format_documents(reranked_docs)
    })
    print('='*60)
    print(f'질문 :{query}\n')
    print(f'표준화된 질문 :{normalized_query}\n')
    print('='*60)
    print(f'답변 :{answer}\n')
    print(f'참고 조항 :{referenced_articles}\n')
    print('**위 답변은 AI에 의해 작성된 답변이므로 약간의 차이가 날 수 있습니다.')

# 7. 사용예시

In [74]:
ask_with_reference("연봉 5천만원인 직장인 소득세는 얼마예요?")

질문 :연봉 5천만원인 직장인 소득세는 얼마예요?

표준화된 질문 :연봉 5천만원인 근로소득이 있는 거주자 소득세는 얼마예요?

답변 :연봉 5천만원인 근로소득에 대한 거주자의 소득세를 계산하기 위해서는 먼저 종합소득산출세액을 계산해야 합니다.

1. 총급여액이 5천만원이므로, 1,400만원 초과 5,000만원 이하 구간에 해당합니다.
2. 이 구간의 세율에 따라서 계산하면:
   - 기본세액 = 84만원 + (5,000만원 - 1,400만원) × 15% 
   - = 84만원 + (3,600만원 × 0.15)
   - = 84만원 + 540만원 
   - = 624만원 

3. 다음으로 근로소득세액공제를 적용합니다. 총급여액이 5천만원이므로, 공제액은 다음과 같이 계산할 수 있습니다:
   - 공제액 = 74만원 - [(5천만원 - 3,300만원) × 8/1000]
   - = 74만원 - [(1,700만원) × 0.008]
   - = 74만원 - 13.6만원 
   - = 60.4만원 (이 금액이 66만원보다 작으므로 66만원으로 설정)
   
4. 따라서 소득세는 624만원 - 66만원 = 558만원이 됩니다.

결론적으로, 연봉 5천만원인 근로소득이 있는 거주자 소득세는 약 558만원입니다.

참고 조항 :소득세법제43조,제51조,제54조,제55조,제59조,제91조,제134조,제146조

**위 답변은 AI에 의해 작성된 답변이므로 약간의 차이가 날 수 있습니다.


In [75]:
ask_with_reference("사업소득이 있는 경우 필요경비는 어떻게 계산하나요?")

질문 :사업소득이 있는 경우 필요경비는 어떻게 계산하나요?

표준화된 질문 :사업소득이 있는 경우 필요경비는 어떻게 계산하나요?

답변 :사업소득이 있는 경우 필요경비는 해당 과세기간의 총수입금액에 대응하는 통상적인 비용의 합계액으로 계산합니다. 특히 과세기간 전의 총수입금액에 대응하는 비용 중 그 과세기간에 확정된 것만 필요경비로 인정됩니다. 필요경비에 대한 세부사항은 대통령령에 의해 정해지며, 이를 통해 필요경비의 범위 및 계산에 필요한 세부 지침이 규정됩니다.

참고 조항 :소득세법제21조,제27조,제28조,제29조,제30조,제31조,제32조,제34조,제35조,제36조,제37조,제52조,제62조,제106조,제119조,제121조,제158조,제159조,제160조,제163조

**위 답변은 AI에 의해 작성된 답변이므로 약간의 차이가 날 수 있습니다.
